In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
empresas = pd.read_excel(r'C:\Users\jober\Downloads\Empresas_registradas_co.xlsx', sheet_name='Sheet1')

In [4]:
# The general information of the dataframe is
empresas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1239 entries, 0 to 1238
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Tipo Documento        1239 non-null   object 
 1   Número Documento      1239 non-null   object 
 2   Razón Social          1239 non-null   object 
 3   Naturaleza            1239 non-null   object 
 4   Tipo Empresa          1239 non-null   object 
 5   Actividad Económica   1174 non-null   object 
 6   Sector                1173 non-null   object 
 7   Tamaño                1239 non-null   object 
 8   Código Sede           1160 non-null   float64
 9   Nombre Sede           1160 non-null   object 
 10  Teléfono              1237 non-null   object 
 11  Dirección             1239 non-null   object 
 12  Tipo Sede             1239 non-null   object 
 13  Ciudad                1239 non-null   object 
 14  Departamento          1239 non-null   object 
 15  Nombre Administrador 

In [5]:
# The number of cells with null data is
empresas.isna().sum()

Tipo Documento           0
Número Documento         0
Razón Social             0
Naturaleza               0
Tipo Empresa             0
Actividad Económica     65
Sector                  66
Tamaño                   0
Código Sede             79
Nombre Sede             79
Teléfono                 2
Dirección                0
Tipo Sede                0
Ciudad                   0
Departamento             0
Nombre Administrador     2
Email Administrador      2
Fecha registro           0
Canal de Registro        0
Prestador Aprobó         2
Agencia Aprobó           2
Agente Aprobó            3
País                     0
dtype: int64

In [6]:
# Typing column the names 
empresas.columns = empresas.columns.str.lower()
empresas.columns = empresas.columns.str.replace(" ","_")

In [7]:
empresas.columns

Index(['tipo_documento', 'número_documento', 'razón_social', 'naturaleza',
       'tipo_empresa', 'actividad_económica', 'sector', 'tamaño',
       'código_sede', 'nombre_sede', 'teléfono', 'dirección', 'tipo_sede',
       'ciudad', 'departamento', 'nombre_administrador', 'email_administrador',
       'fecha_registro', 'canal_de_registro', 'prestador_aprobó',
       'agencia_aprobó', 'agente_aprobó', 'país'],
      dtype='object')

In [8]:
# Cleaning NaN data. Data as 'nan' are not actual "NaN":
for column in empresas.columns:
    empresas[column] = empresas[column].replace('nan', pd.NA)
    empresas[column] = empresas[column].replace('', pd.NA)


In [9]:
for col in empresas.columns:
    if empresas[col].dtype == 'object':
        empresas[col] = empresas[col].astype(str)
        empresas[column] = [str(i).lower() for i in empresas[column]]
        empresas[column] = [str(i).strip() for i in empresas[column]]

In [10]:
# subset_columns = ['tipo_documento', 'número_documento', 'razón_social', 'naturaleza',
#        'tipo_empresa', 'actividad_económica', 'sector', 'tamaño',
#        'nombre_sede', 'dirección', 'tipo_sede','ciudad', 'departamento',
#        'nombre_administrador', 'email_administrador', 'canal_de_registro', 
#        'prestador_aprobó', 'agencia_aprobó', 'agente_aprobó', 'país']

# # Replace empty strings with NaN (optional)
# for column in subset_columns:
#     empresas[column] = [str(i).lower() for i in empresas[column]]
#     empresas[column] = [str(i).strip() for i in empresas[column]]

In [11]:
# Modiying to datetime 
# empresas['fecha_registro'] = pd.to_datetime(empresas['fecha_registro'])

empresas['fecha_registro'] = empresas['fecha_registro'].fillna("")

# Function to handle different date formats
def parse_dates(date_str):
    if pd.isna(date_str) or date_str == "":  # Handle empty strings or NaN
        return pd.NaT
    
    date_str = str(date_str).strip()
    
    # Case 1: Excel serial number
    if date_str.isdigit():
        return pd.to_datetime(int(date_str), origin='1899-12-30', unit='D')

    # Case 2: DD-MM-YYYY format
    try:
        return pd.to_datetime(date_str, format="%d-%m-%Y")
    except ValueError:
        pass  # If it fails, try the next method
    
    # Case 3: Other datetime formats
    return pd.to_datetime(date_str, errors='coerce', dayfirst=True)  

# Convert date columns
empresas['fecha_registro'] = empresas['fecha_registro'].apply(parse_dates)

# Floor the date to remove time (ensures it's still a datetime object)
empresas['fecha_registro'] = empresas['fecha_registro'].dt.floor('D')

# Convert to string format for final output
# empresas['fecha_registro'] = empresas['fecha_registro'].dt.strftime('%Y-%m-%d')


C:\Users\jober\AppData\Local\Temp\ipykernel_14564\3597121015.py:24: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(date_str, errors='coerce', dayfirst=True)


In [12]:
non_numeric = empresas[empresas['teléfono'].astype(str).str.contains(r'\D')]
print(f'The number of non numerical values in telephone column is:', non_numeric['teléfono'].count())
print(non_numeric['teléfono'])

The number of non numerical values in telephone column is: 32
2                    nan
3                    nan
12             \t3785184
16             \t3366000
222            o 3799622
242         57 1 6672200
247            326 72 60
277      3 1 6 4 7 2 2 8
285     3853162 ext 3042
291       3135252301- 36
324      4488511 EXT 119
330          313 5759484
352         310 734 9503
363          310 2460764
375         (1)\t5833300
390         035-385-9179
429         315 497 0774
464         57 1 7050000
760          605/3059838
891      3853100 Ext 817
904          320 3004854
986     +57 605 371-0707
1015         316 8316441
1056        (605)3854616
1057        605385 36 48
1078         310 2917909
1111       +573164929864
1115     6 0 5 4 0 3 2 9
1144         315 8518637
1149         310 2499087
1155         323 3418230
1185         316 3182293
Name: teléfono, dtype: object


In [13]:
# Remove all non-digit characters except spaces
empresas['teléfono_clean'] = empresas['teléfono'].astype(str).str.replace(r'[^\d ]', '', regex=True)

# Remove all spaces
empresas['teléfono_clean'] = empresas['teléfono_clean'].str.replace(' ', '')

# Then extract only the first group of digits
empresas['teléfono_clean'] = empresas['teléfono_clean'].str.extract(r'^(\d+)', expand=False)

# Convert to integer
empresas['teléfono_clean'] = empresas['teléfono_clean'].astype('Int64')

In [14]:
# Creating the data pionts
empresas['mes'] = empresas['fecha_registro'].dt.month
empresas['año'] = empresas['fecha_registro'].dt.year

In [15]:
# The new companies for this months are
today = datetime.today()

if today.month == 1:
    prev_month = 12
    prev_year = today.year - 1
else:
    prev_month = today.month - 1
    prev_year = today.year

# The filtered registries are: 
filter = (empresas['mes'] == prev_month) & (empresas['año'] == prev_year)
print(f'La cantidad de empresas registradas en durante el mes {prev_month} fueron: ', empresas[filter]['tipo_documento'].count())
print(f'Las {empresas[filter]['tipo_documento'].count()} empresas registradas fueron: ', empresas[filter]['razón_social'])
print()
empresas = empresas[filter]

La cantidad de empresas registradas en durante el mes 9 fueron:  8
Las 8 empresas registradas fueron:  1196                           ACTIVOS S.A.S.
1212                         OFIEXPORT S.A.S.
1233                     AMROP TOP MANAGEMENT
1234                          SUNNY APP S.A.S
1235    LAVANDERIA MARITIMA DEL CARIBE S.A.S.
1236             SSC TELESALES COMPANY S.A.S.
1237                     ANDREA PEREZ NAVARRO
1238       CONSORCIO MANTENIMIENTO ESCOLAR BQ
Name: razón_social, dtype: object



In [16]:
# filter = (empresas['año'] <= 2024)
# empresas = empresas[filter]
# empresas.head()

In [17]:
# Exporting the data
# empresas.to_parquet(f'empresas_2024+anteriores', compression='zstd')
empresas.to_parquet(f'empresas_{prev_year}_{prev_month}', compression='zstd')